# SNNAI v6-5-4 DDP Multi-GPU SNN LM + Transformer Comparison

This notebook runs the v6.5.4 multi-GPU DDP training script.

- v6.5.0: sequence-axis recurrence, positional encoding, nn.Embedding
- v6.5.1-v6.5.3: DataParallel attempts (incompatible with snntorch LIF)
- v6.5.4: DistributedDataParallel via torch.multiprocessing.spawn
  (each process owns one GPU + one model replica with correct LIF state).

In [ ]:
# Install dependencies
# CUDA 12.6 build keeps P100 (sm_60) and T4 (sm_75) compatibility
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu126
!pip install -q snntorch numpy requests

In [ ]:
# Clone SNNAI repository
!rm -rf snnai
!git clone --depth 1 --branch v6.5.4 https://github.com/sabunosuke1008-create/snnai.git
import sys
sys.path.insert(0, 'snnai')

In [ ]:
# Launch DDP training (uses both T4 GPUs via mp.spawn + DistributedDataParallel).
# The script downloads corpus, trains BPE tokenizer, and trains SNN+Transformer.
%run train_ddp.py --epochs 8 --batch_size 16 --seq_len 128

In [ ]:
# Load and report final training metrics
import json, os, torch
from snnai.benchmarks.parallel_utils import unwrap_model
from snnai.modules.language.large_scale_lm import LargeScaleSNNLM
from snnai.benchmarks.transformer_comparison import TransformerBaseline

history_path = '/kaggle/working/snnai_v6_ddp_history.json'
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    print('SNN final val ppl:', history['snn']['val_ppl'][-1])
    print('SNN final train ppl:', history['snn']['train_ppl'][-1])
    print('Transformer final val ppl:', history['transformer']['val_ppl'][-1])
    print('Transformer final train ppl:', history['transformer']['train_ppl'][-1])
    print('SNN per-epoch:', list(zip(history['snn']['train_loss'], history['snn']['val_ppl'])))
    print('TX per-epoch:', list(zip(history['transformer']['train_loss'], history['transformer']['val_ppl'])))
else:
    print('History file not found; training may have failed')

In [ ]:
# Quick latency / energy / parameter comparison (single-GPU inference)
import torch, time, json
from snnai.modules.language import BPETokenizer
from snnai.modules.language.large_scale_lm import LargeScaleSNNLM
from snnai.benchmarks.transformer_comparison import TransformerBaseline, compare_models
from snnai.benchmarks.energy_quantification import quantize_energy

# Reconstruct tokenizer from the saved pickle
import pickle
with open('/kaggle/working/bpe.pkl', 'rb') as f:
    tokenizer = pickle.load(f)
vocab_size = tokenizer.vocab_size

# Load saved checkpoints (trained on both GPUs, infer on GPU 0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load('/kaggle/working/snnai_v6_ddp.pt', map_location=device, weights_only=True)

snn_model = LargeScaleSNNLM(vocab_size=vocab_size, embed_dim=128, hidden_dim=512,
                            num_layers=3, dropout=0.2, output_mode='mem_last',
                            use_sequence_recurrent=True, use_positional_encoding=True,
                            max_seq_len=128).to(device)
snn_model.load_state_dict(ckpt['snn_state_dict'])
transformer_model = TransformerBaseline(vocab_size=vocab_size, d_model=256,
                                        nhead=4, num_layers=4, dim_feedforward=1024).to(device)
transformer_model.load_state_dict(ckpt['tx_state_dict'])

seq_len, time_steps = 128, 20
sample_input = torch.zeros(time_steps, 1, seq_len, vocab_size).to(device)
comparison = compare_models(snn_model, transformer_model, sample_input)
print(json.dumps({k: v for k, v in comparison.items()}, indent=2))

# Energy report
sample_text = 'ROMEO:'
sample_ids = tokenizer.encode(sample_text)[:seq_len]
sample_ids = sample_ids + [0] * (seq_len - len(sample_ids))
sample_one_hot = torch.zeros(1, seq_len, vocab_size)
sample_one_hot.scatter_(2, torch.tensor([sample_ids]).unsqueeze(2), 1.0)
sample_input_energy = sample_one_hot.unsqueeze(0).repeat(time_steps, 1, 1, 1).to(device)
energy = quantize_energy(snn_model, sample_input_energy, time_steps=time_steps)
print('\nSNN energy report:')
print(json.dumps(energy, indent=2))

In [ ]:
# Generate sample text (sampling with temperature/top-k/top-p)
from snnai.benchmarks.generation_metrics import evaluate_generation
snn_model.eval()
sampled = evaluate_generation(
    snn_model, tokenizer, ['ROMEO:', 'JULIET:', 'The '],
    max_chars=200, device=device,
    temperature=0.8, top_k=10, top_p=0.9, do_sample=True,
    repetition_penalty=1.5, penalty_window=16,
    generation_bias_tokens=None,
    generation_bias_weight=0.0,
)
print('Sampled generation:')
for g in sampled['generated']:
    print(g[:200])